In [ ]:
# !!! ATTENZIONE !!!
# modificare i path
annotations_path = "/inserisci/il/path/per/pp4av/annotations"
images_path = "/inserisci/il/path/per/pp4av/images"

## Analisi Esplorativa dei Dati (EDA) del Dataset PP4AV
Poiché PP4AV raccoglie frame video da diverse città e con diverse ottiche (incluse lenti *fisheye*), le immagini e i target presentano una forte eterogeneità.

Il seguente script utilizza `Pandas`, `Numpy`, `Matplotlib`, `Seaborn` e `PIL` per analizzare le annotazioni YOLO (`.txt`) e le immagini originali (`.png` / `.jpg`), estraendo metriche cruciali per l'addestramento:

1. **Distribuzione per scenario e classe:** per valutare il *class imbalance* tra volti e targhe e la loro frequenza nei vari contesti (es. guida notturna o diurna).
2. **Risoluzioni native:** per identificare le dimensioni originali delle immagini e scegliere coerentemente il parametro `imgsz` per il *letterboxing* di YOLO.
3. **Area relativa (Small Object Detection):** calcola quale percentuale del frame è occupata dai *bounding box* (su scala logaritmica), evidenziando la criticità dei target microscopici.
4. **Aspect ratio:** analizza il rapporto larghezza/altezza per capire la varianza di forma degli oggetti da rilevare.

Lo script genererà prima un **report testuale** con le statistiche descrittive e, a seguire, **5 grafici interattivi** per la visualizzazione dei dati.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

def eda_pp4av(images_dir, labels_dir):
    print("=== AVVIO EDA ===\n")
    
    # --- STRUTTURE DATI ---
    dati_immagini = []
    dati_oggetti = []
    
    scenari = [d for d in os.listdir(labels_dir) if os.path.isdir(os.path.join(labels_dir, d))]
    
    # --- 1. LETTURA ESTRAZIONE DATI ---
    for scenario in scenari:
        percorso_labels = os.path.join(labels_dir, scenario)
        percorso_immagini = os.path.join(images_dir, scenario)
        
        file_txt = glob.glob(os.path.join(percorso_labels, "*.txt"))
        
        for txt in file_txt:
            nome_base = os.path.splitext(os.path.basename(txt))[0]
            
            img_path = os.path.join(percorso_immagini, nome_base + ".png")
            if not os.path.exists(img_path):
                img_path = os.path.join(percorso_immagini, nome_base + ".jpg")
                
            img_w, img_h = 0, 0
            if os.path.exists(img_path):
                with Image.open(img_path) as img:
                    img_w, img_h = img.size
                    
            num_oggetti = 0
            with open(txt, "r") as f:
                linee = f.readlines()
                num_oggetti = len(linee)
                
                for linea in linee:
                    if linea.strip():
                        parti = linea.split()
                        classe_id = int(parti[0])
                        nome_classe = "Volto" if classe_id == 0 else "Targa"
                        
                        x_c, y_c, w_norm, h_norm = map(float, parti[1:5])
                        area_relativa = w_norm * h_norm
                        aspect_ratio = w_norm / h_norm if h_norm > 0 else 0
                        
                        dati_oggetti.append({
                            "Scenario": scenario.capitalize(),
                            "Classe": nome_classe,
                            "Area_Relativa": area_relativa,
                            "Aspect_Ratio": aspect_ratio
                        })
            
            dati_immagini.append({
                "Scenario": scenario.capitalize(),
                "Risoluzione": f"{img_w}x{img_h}",
                "Totale_Oggetti": num_oggetti
            })

    # Creazione DataFrame
    df_immagini = pd.DataFrame(dati_immagini)
    df_oggetti = pd.DataFrame(dati_oggetti)
    
    # --- 2. GENERAZIONE REPORT TESTUALE ---
    print("--------------------------------------------------")
    print("  REPORT STATISTICO DEL DATASET PP4AV")
    print("--------------------------------------------------\n")

    # A. Statistiche Generali Immagini
    tot_immagini = len(df_immagini)
    tot_annotazioni = len(df_oggetti)
    print(f"1. DIMENSIONI DEL DATASET:")
    print(f"   - Totale Immagini: {tot_immagini}")
    print(f"   - Totale Annotazioni (Bounding Box): {tot_annotazioni}")
    print(f"   - Media oggetti per immagine: {df_immagini['Totale_Oggetti'].mean():.2f}")
    print(f"   - Massimo oggetti in una singola immagine: {df_immagini['Totale_Oggetti'].max()}\n")

    # B. Distribuzione Risoluzioni
    print(f"2. RISOLUZIONI DELLE IMMAGINI ORIGINALI:")
    risoluzioni_counts = df_immagini["Risoluzione"].value_counts()
    for res, count in risoluzioni_counts.items():
        percentuale = (count / tot_immagini) * 100
        print(f"   - {res}: {count} immagini ({percentuale:.1f}%)")
    print("\n")

    # C. Distribuzione Classi
    print(f"3. DISTRIBUZIONE DELLE CLASSI:")
    classi_counts = df_oggetti["Classe"].value_counts()
    for classe, count in classi_counts.items():
        percentuale = (count / tot_annotazioni) * 100
        print(f"   - {classe}: {count} annotazioni ({percentuale:.1f}%)")
    print("\n")

    # D. Metriche Geometriche (Area e Aspect Ratio)
    print(f"4. METRICHE GEOMETRICHE DEI BOUNDING BOX:")
    for classe in ["Volto", "Targa"]:
        df_classe = df_oggetti[df_oggetti["Classe"] == classe]
        print(f"   --- {classe.upper()} ---")
        
        area_media = df_classe["Area_Relativa"].mean() * 100 
        area_min = df_classe["Area_Relativa"].min() * 100
        area_max = df_classe["Area_Relativa"].max() * 100
        print(f"   Area Media rispetto al frame: {area_media:.4f}% (Min: {area_min:.4f}%, Max: {area_max:.2f}%)")
        
        ar_medio = df_classe["Aspect_Ratio"].mean()
        ar_min = df_classe["Aspect_Ratio"].min()
        ar_max = df_classe["Aspect_Ratio"].max()
        print(f"   Aspect Ratio Medio (W/H): {ar_medio:.2f} (Min: {ar_min:.2f}, Max: {ar_max:.2f})\n")

    print("--------------------------------------------------")
    print("  FINE REPORT - Generazione grafici in corso...")
    print("--------------------------------------------------\n")

    # --- 3. GENERAZIONE GRAFICI (MATPLOTLIB & SEABORN) ---
    sns.set_theme(style="whitegrid")
    colore_volti = "#1A4D94"
    colore_targhe = "#7EC8E3"
    palette_classi = {"Volto": colore_volti, "Targa": colore_targhe}

    # --- GRAFICO 0: Distribuzione Globale delle Classi ---
    plt.figure(figsize = (8, 6))
    conteggio_classi = df_oggetti["Classe"].value_counts().reset_index()
    conteggio_classi.columns = ["Classe", "Conteggio"]
    
    ax0 = sns.barplot(data = conteggio_classi, x = "Classe", y = "Conteggio", hue = "Classe", palette = palette_classi, legend = False)
    ax0.bar_label(ax0.containers[0], padding = 3, fontsize = 12)
    
    plt.title("0. Distribuzione Globale delle Classi nel Dataset PP4AV", fontsize=14, pad=15)
    plt.xlabel("Classe di Oggetti", fontsize=12)
    plt.ylabel("Numero Totale di Annotazioni", fontsize=12)
    plt.ylim(0, conteggio_classi["Conteggio"].max() * 1.15)
    plt.tight_layout()
    plt.show()

    # --- GRAFICO 1: Distribuzione per Scenario ---
    plt.figure(figsize = (12, 6))
    sns.countplot(data = df_oggetti, x = "Scenario", hue = "Classe", palette = palette_classi)
    plt.title("1. Distribuzione per Scenario", fontsize = 14, pad = 15)
    plt.xlabel("Scenario Operativo", fontsize = 12)
    plt.ylabel("Conteggio", fontsize = 12)
    plt.xticks(rotation = 45)
    plt.tight_layout()
    plt.show()

In [ ]:
# ESECUZIONE EDA
eda_pp4av(images_path, annotations_path)